# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install --quiet mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load dataset from the Croissant schema
dataset = mlc.Dataset(croissant_url)

metadata = dataset.metadata
print(f"Dataset Name: {metadata.name}")
print(f"Description: {metadata.description}")
print(f"Version: {getattr(metadata, 'version', 'N/A')}")
print(f"Identifier: {getattr(metadata, 'identifier', 'N/A')}")

## 2. Data Overview
Review available record sets and their `@id`s and fields.

We first inspect the available record sets, fields, and relevant identifiers. You should use these IDs in future steps.

In [ ]:
# List all record sets with their @id, name, and fields
record_sets = list(dataset.metadata.record_sets)
print(f'Found {len(record_sets)} record sets:')
for record_set in record_sets:
    print(f"- RecordSet @id: {record_set.id}, name: {getattr(record_set, 'name', 'N/A')}")
    if hasattr(record_set, 'fields'):
        fields_list = list(record_set.fields)
        for field in fields_list:
            print(f"    - Field @id: {field.id}, name: {getattr(field, 'name', 'N/A')}, dataType: {getattr(field, 'data_type', 'N/A')}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

Below, we extract data from all available record sets (by their @id), storing them in a DataFrame for each record set.

In [ ]:
# Compile all record set @id's
record_set_ids = [rs.id for rs in dataset.metadata.record_sets]
print("All RecordSet @id's:", record_set_ids)

dataframes = {}  # For each RecordSet @id, a corresponding pandas DataFrame

for record_set_id in record_set_ids:
    print(f"Loading records for RecordSet @id: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"  - Loaded {len(df)} records. Columns: {df.columns.tolist()}")

# Inspect columns of the first record set as an example
if record_set_ids:
    first_rs_id = record_set_ids[0]
    print(f"\nColumns for first RecordSet (@id = {first_rs_id}):")
    print(dataframes[first_rs_id].columns.tolist())
    dataframes[first_rs_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

Let's select a record set with at least one numeric field.

In [ ]:
# Choose a record set with suitable numeric fields
# (Replace these IDs as appropriate for your exploration)
import numpy as np

# We'll auto-detect the first numeric field in the first record set (if available)
if record_sets:
    record_set = record_sets[0]
    record_set_id = record_set.id
    df = dataframes[record_set_id]

    # Auto-detect a numeric field (integer/float)
    numeric_field = None
    for field in record_set.fields:
        if getattr(field, 'data_type', None) in ['schema:Float', 'schema:Integer', 'http://schema.org/Float', 'http://schema.org/Integer']:
            # Look for field @id in DataFrame columns
            col_id = field.id
            if col_id in df.columns:
                numeric_field = col_id
                print(f"Numeric field detected for EDA: {field.id} ({field.name})")
                break

    if numeric_field is not None:
        # Convert to numeric in case of string
        df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')
        threshold = df[numeric_field].mean() if not np.isnan(df[numeric_field].mean()) else 0
        filtered_df = df[df[numeric_field] > threshold].copy()
        print(f"Filtered records in '{record_set_id}' with {numeric_field} > {threshold:.2f}:")
        display(filtered_df.head())

        # Normalization
        if df[numeric_field].std() > 0:
            filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - df[numeric_field].mean()) / df[numeric_field].std()
            print(f"Normalized {numeric_field} for filtered records:")
            display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())
        else:
            print("Standard deviation is zero, skipping normalization.")

        # Try to choose a categorical/group field
        group_field = None
        for field in record_set.fields:
            if getattr(field, 'data_type', None) in ['schema:Text', 'http://schema.org/Text', None] and field.id in df.columns:
                group_field = field.id
                break

        if group_field is not None:
            grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
            print(f"Grouped data by '{group_field}':")
            display(grouped_df.head())
        else:
            print("No suitable group field was found.")
    else:
        print("No numeric field detected in this RecordSet for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. Let's create basic plots if numeric data are present.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualize the numeric field if it exists
if record_sets and numeric_field is not None:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field].dropna(), kde=True)
    plt.title(f"Distribution of '{numeric_field}' in '{record_set_id}' RecordSet")
    plt.xlabel(numeric_field)
    plt.show()

    if group_field is not None:
        plt.figure(figsize=(10,5))
        sns.boxplot(x=group_field, y=numeric_field, data=df)
        plt.xticks(rotation=45)
        plt.title(f"{numeric_field} grouped by {group_field}")
        plt.show()

## 6. Conclusion
In this notebook, we've demonstrated how to explore a FAIR^2-conformant biological dataset using the `mlcroissant` library. We:
- Loaded the dataset and inspected its metadata and structure (record sets and fields), referencing all entities by their `@id`.
- Extracted data using record set `@id`s into pandas DataFrames.
- Performed basic exploratory data analysis (filtering, normalization, grouping).
- Visualized numeric data distributions and categorical breakdowns.

Explore further by adjusting criteria, inspecting additional record sets, or using more advanced analytics and visualizations!